In [ ]:
# ============================================================
# BLOCK 5: TIME SERIES MODELS (SARIMA)
# Project: Forecasting Household Deposit Volume in Russia
# Author: Nadezhda Silkina
# Date: 2026
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries loaded")

# ============================================================
# 2. LOAD DATA
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"✅ Data loaded. Records: {len(df)}")
print(f"Period: from {df.index.min()} to {df.index.max()}")

# ============================================================
# 3. METHODOLOGICAL JUSTIFICATION FOR CHOOSING SARIMA
# ============================================================

print("\n" + "="*60)
print("3. WHY SARIMA AND NOT PROPHET")
print("="*60)

print("""
📌 KEY PROBLEM: 4TH-ORDER AUTOCORRELATION OF RESIDUALS

In all previous models (baseline Ridge, full model with 38 features)
residual diagnostics on the training sample revealed significant 4th-order
autocorrelation (Breusch-Godfrey test, p < 0.05).

WHY NOT PROPHET:
- Prophet does not directly account for autocorrelation in residuals
- Produces biased forecasts with autocorrelated errors

WHY SARIMA:
- Explicitly models autocorrelation through AR/MA components
- Seasonal components can capture quarterly/annual structure
- Ljung-Box test allows checking whether autocorrelation is eliminated

CONCLUSION: We choose SARIMA to solve the autocorrelation problem.
""")

# ============================================================
# 4. STATIONARITY ANALYSIS OF DEPOS SERIES
# ============================================================

print("\n" + "="*60)
print("4. STATIONARITY ANALYSIS OF DEPOS SERIES")
print("="*60)

print("""
📌 SERIES UNDER TEST: DEPOS (household deposit volume in Russia, billion RUB)

Two tests with different hypotheses:
- ADF: H₀ = series is NON-stationary (has unit root)
- KPSS: H₀ = series is stationary (no unit root)
""")

# 4.1. ADF test (constant, no trend)
print("\n🔍 Dickey-Fuller test (ADF):")
print("   H₀: series is non-stationary | H₁: series is stationary")
adf_result = adfuller(df['DEPOS'], autolag='AIC', regression='c')
print(f"   ADF statistic: {adf_result[0]:.4f}")
print(f"   p-value: {adf_result[1]:.4f}")
print(f"   Critical values: 1%: {adf_result[4]['1%']:.4f}, 5%: {adf_result[4]['5%']:.4f}")
print(f"   Conclusion: {'Stationary ✅' if adf_result[1] < 0.05 else 'Non-stationary ❌ (p > 0.05, do not reject H₀)'}")

# 4.2. KPSS test (constant + trend)
print("\n🔍 KPSS test:")
print("   H₀: series is stationary | H₁: series is non-stationary")
kpss_result = kpss(df['DEPOS'], regression='ct')
print(f"   KPSS statistic: {kpss_result[0]:.4f}")
print(f"   p-value: {kpss_result[1]:.4f}")
print(f"   Critical values: 1%: {kpss_result[3]['1%']:.4f}, 5%: {kpss_result[3]['5%']:.4f}")
print(f"   Conclusion: {'Stationary ✅' if kpss_result[1] > 0.05 else 'Non-stationary ❌ (p < 0.05, reject H₀)'}")

print("\n📌 OVERALL CONCLUSION: Both tests confirm that DEPOS series is NON-STATIONARY")
print("   → Differencing is required")

# 4.3. Series plot
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['DEPOS'], linewidth=2, color='steelblue')
plt.title('DEPOS Series: Household Deposit Volume in Russia', fontsize=14)
plt.xlabel('Date')
plt.ylabel('billion RUB')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_depos_series.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 5. DETERMINING THE ORDER OF DIFFERENCING (d)
# ============================================================

print("\n" + "="*60)
print("5. DETERMINING THE ORDER OF DIFFERENCING (d)")
print("="*60)

print("""
📌 METHODOLOGY: Sequentially check series differences for stationarity.
d=0 → if non-stationary, d=1 → if non-stationary, d=2.
""")

# --- Check first difference (d=1) ---
print("\n🔍 Checking FIRST difference (d=1):")
df['DEPOS_diff1'] = df['DEPOS'].diff()

adf_diff1 = adfuller(df['DEPOS_diff1'].dropna(), autolag='AIC', regression='c')
kpss_diff1 = kpss(df['DEPOS_diff1'].dropna(), regression='c')

print(f"   ADF: stat = {adf_diff1[0]:.4f}, p = {adf_diff1[1]:.4f}")
print(f"   KPSS: stat = {kpss_diff1[0]:.4f}, p = {kpss_diff1[1]:.4f}")

d1_stationary = (adf_diff1[1] < 0.05) and (kpss_diff1[1] > 0.05)

if d1_stationary:
    print(f"   ✅ First difference is STATIONARY → d = 1")
    d_order = 1
else:
    print(f"   ❌ First difference is NON-STATIONARY (ADF p={adf_diff1[1]:.4f} > 0.05)")
    print(f"   → Checking second difference")

    # --- Check second difference (d=2) ---
    print("\n🔍 Checking SECOND difference (d=2):")
    df['DEPOS_diff2'] = df['DEPOS_diff1'].diff()

    adf_diff2 = adfuller(df['DEPOS_diff2'].dropna(), autolag='AIC', regression='c')
    kpss_diff2 = kpss(df['DEPOS_diff2'].dropna(), regression='c')

    print(f"   ADF: stat = {adf_diff2[0]:.4f}, p = {adf_diff2[1]:.4f}")
    print(f"   KPSS: stat = {kpss_diff2[0]:.4f}, p = {kpss_diff2[1]:.4f}")

    d2_stationary = (adf_diff2[1] < 0.05) and (kpss_diff2[1] > 0.05)

    if d2_stationary:
        print(f"   ✅ Second difference is STATIONARY → d = 2")
        d_order = 2
    else:
        print(f"   ⚠️ Even second difference is non-stationary (unusual for economic series)")
        d_order = 2  # usually sufficient

# Difference plots
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(df.index, df['DEPOS'], linewidth=1.5, color='steelblue')
axes[0].set_title('Original series (d=0)')
axes[0].set_ylabel('DEPOS')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df.index, df['DEPOS_diff1'], linewidth=1.5, color='darkorange')
axes[1].set_title(f'First difference (d=1): ADF p={adf_diff1[1]:.4f}, KPSS p={kpss_diff1[1]:.4f}')
axes[1].set_ylabel('Δ DEPOS')
axes[1].grid(True, alpha=0.3)

if d_order == 2:
    axes[2].plot(df.index, df['DEPOS_diff2'], linewidth=1.5, color='green')
    axes[2].set_title(f'Second difference (d=2): ADF p={adf_diff2[1]:.4f}, KPSS p={kpss_diff2[1]:.4f}')
    axes[2].set_ylabel('Δ² DEPOS')
    axes[2].grid(True, alpha=0.3)
else:
    axes[2].axis('off')

plt.tight_layout()
plt.savefig('05_diff_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 RESULT: Selected differencing order d = {d_order}")

# ============================================================
# 6. ACF AND PACF ANALYSIS
# ============================================================

print("\n" + "="*60)
print("6. ACF AND PACF ANALYSIS")
print("="*60)

print("""
📌 HOW TO READ ACF/PACF PLOTS:
- Shaded area — 95% confidence interval (±1.96/√n)
- Bar extending beyond the area → autocorrelation is significant (p < 0.05)
- Lag 0 is always 1 (series with itself) — ignore it
- Lag 1 is the FIRST bar after lag 0
""")

# Create training sample (last 12 months - test)
train_size = len(df) - 12
depos_train = df['DEPOS'].iloc[:train_size]
depos_test = df['DEPOS'].iloc[train_size:]

print(f"\n📊 Training sample: {len(depos_train)} records")
print(f"📊 Test sample: {len(depos_test)} records")
print(f"📊 Test period: {depos_test.index[0]} — {depos_test.index[-1]}")

# Differenced series (d=2)
depos_train_diff = depos_train.diff().diff().dropna()

# ACF and PACF plots with significance lines
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Significance threshold (95% CI)
n_obs = len(depos_train)
threshold = 1.96 / np.sqrt(n_obs)

# ACF of original series
plot_acf(depos_train, lags=36, ax=axes[0, 0])
axes[0, 0].axhline(y=threshold, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'±1.96/√n = ±{threshold:.3f}')
axes[0, 0].axhline(y=-threshold, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[0, 0].set_title('ACF: Original series (DEPOS, d=0)')
axes[0, 0].set_xlabel('Lag (months)')
axes[0, 0].set_ylabel('Autocorrelation')
axes[0, 0].legend(fontsize=8)

# PACF of original series
plot_pacf(depos_train, lags=36, ax=axes[0, 1])
axes[0, 1].axhline(y=threshold, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'±1.96/√n')
axes[0, 1].axhline(y=-threshold, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[0, 1].set_title('PACF: Original series (DEPOS, d=0)')
axes[0, 1].set_xlabel('Lag (months)')
axes[0, 1].set_ylabel('Partial autocorrelation')
axes[0, 1].legend(fontsize=8)

# ACF of differenced series (d=2)
plot_acf(depos_train_diff, lags=36, ax=axes[1, 0])
axes[1, 0].axhline(y=threshold, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'±1.96/√n')
axes[1, 0].axhline(y=-threshold, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[1, 0].set_title('ACF: Differenced series (d=2)')
axes[1, 0].set_xlabel('Lag (months)')
axes[1, 0].set_ylabel('Autocorrelation')
axes[1, 0].legend(fontsize=8)

# PACF of differenced series (d=2)
plot_pacf(depos_train_diff, lags=36, ax=axes[1, 1])
axes[1, 1].axhline(y=threshold, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'±1.96/√n')
axes[1, 1].axhline(y=-threshold, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[1, 1].set_title('PACF: Differenced series (d=2)')
axes[1, 1].set_xlabel('Lag (months)')
axes[1, 1].set_ylabel('Partial autocorrelation')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('05_acf_pacf_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# ACF/PACF interpretation
print("\n📌 Observations from ACF/PACF:")
print("""
   ORIGINAL SERIES (d=0):
   - ACF: slowly decaying → non-stationarity
   - PACF: significant peak only at lag 1

   DIFFERENCED SERIES (d=2):
   - ACF: significant lags 1, 11, 13, 23, 24...
   - PACF: significant lags 1, 2, 3, 4, 11

   CONCLUSION: Presence of significant lags 11, 13, 23, 24 in ACF indicates
   ANNUAL seasonality (s=12). Quarterly seasonality (s=4)
   is not confirmed.
""")

# ============================================================
# 7. AUTOMATIC SARIMA PARAMETER SELECTION
# ============================================================

print("\n" + "="*60)
print("7. AUTOMATIC SARIMA PARAMETER SELECTION")
print("="*60)

def evaluate_sarima_model(order, seasonal_order, train_data, test_data):
    """
    Trains SARIMA model and evaluates quality + residual diagnostics.
    """
    try:
        model = SARIMAX(
            train_data,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        results = model.fit(disp=False)

        # Forecast
        forecast = results.forecast(steps=len(test_data))

        # Test metrics
        r2_test = r2_score(test_data, forecast)
        rmse_test = np.sqrt(mean_squared_error(test_data, forecast))
        mae_test = mean_absolute_error(test_data, forecast)

        # Residual diagnostics (Ljung-Box)
        residuals = results.resid
        lb_test = acorr_ljungbox(residuals, lags=[4, 8, 12], return_df=True)
        lb_min_p = lb_test['lb_pvalue'].min()

        # Check: residuals not autocorrelated?
        residuals_ok = lb_min_p > 0.05

        return {
            'model': results,
            'forecast': forecast,
            'r2_test': r2_test,
            'rmse_test': rmse_test,
            'mae_test': mae_test,
            'aic': results.aic,
            'bic': results.bic,
            'lb_min_p': lb_min_p,
            'residuals_ok': residuals_ok,
            'success': True
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

# Define parameter grid
# Considering annual seasonality s=12 (from ACF/PACF)
p_range = range(0, 4)  # 0-3
d = d_order  # = 2
q_range = range(0, 4)  # 0-3
P_range = range(0, 2)  # 0-1
D_range = range(0, 2)  # 0-1
Q_range = range(0, 2)  # 0-1
s = 12  # ANNUAL seasonality (from ACF/PACF)

print(f"\n🔍 Starting grid search...")
print(f"   Non-seasonal: p={list(p_range)}, d={[d]}, q={list(q_range)}")
print(f"   Seasonal: P={list(P_range)}, D={list(D_range)}, Q={list(Q_range)}, s={s}")
print(f"   Total combinations: {len(p_range) * len(q_range) * len(P_range) * len(D_range) * len(Q_range)}")
print(f"\n⌛ Searching for models with uncorrelated residuals may take several minutes...")

all_results = []

for p, q, P, D, Q in product(p_range, q_range, P_range, D_range, Q_range):
    order = (p, d, q)
    seasonal_order = (P, D, Q, s)

    result = evaluate_sarima_model(order, seasonal_order, depos_train, depos_test)

    if result['success']:
        all_results.append({
            'order': order,
            'seasonal_order': seasonal_order,
            'r2_test': result['r2_test'],
            'rmse_test': result['rmse_test'],
            'mae_test': result['mae_test'],
            'aic': result['aic'],
            'bic': result['bic'],
            'lb_min_p': result['lb_min_p'],
            'residuals_ok': result['residuals_ok'],
            'result': result
        })

print(f"\n📊 Grid search completed. Found {len(all_results)} models.")

# Models with "good" residuals (Ljung-Box p > 0.05)
good_residuals = [m for m in all_results if m['residuals_ok']]
print(f"\n📊 Models with uncorrelated residuals (Ljung-Box p > 0.05): {len(good_residuals)}")

if good_residuals:
    print("\n🏆 Models with uncorrelated residuals (by AIC):")
    good_by_aic = sorted(good_residuals, key=lambda x: x['aic'])
    for i, model in enumerate(good_by_aic[:5], 1):
        print(f"   {i}. SARIMA{model['order']}×{model['seasonal_order']}: "
              f"AIC={model['aic']:.2f}, R²_test={model['r2_test']:.4f}, "
              f"LB p={model['lb_min_p']:.4f}")

    best_model_info = good_by_aic[0]
    print(f"\n✅ SELECTION: Model with uncorrelated residuals and minimum AIC")
else:
    print("\n⚠️ No model produced uncorrelated residuals!")
    print("   → Selecting model with maximum Ljung-Box p-value (least autocorrelated residuals)")

    best_by_lb = max(all_results, key=lambda x: x['lb_min_p'])
    print(f"   → Model: SARIMA{best_by_lb['order']}×{best_by_lb['seasonal_order']}, "
          f"LB p={best_by_lb['lb_min_p']:.4f}")
    best_model_info = best_by_lb

best_order = best_model_info['order']
best_seasonal_order = best_model_info['seasonal_order']
best_result = best_model_info['result']

print(f"\n✅ Selected model: SARIMA{best_order}×{best_seasonal_order}")
print(f"   AIC = {best_model_info['aic']:.2f}")
print(f"   BIC = {best_model_info['bic']:.2f}")
print(f"   R²_test = {best_model_info['r2_test']:.4f}")
print(f"   RMSE_test = {best_model_info['rmse_test']:.2f} billion RUB")
print(f"   Ljung-Box min p = {best_model_info['lb_min_p']:.4f}")
print(f"   Residuals: {'✅ Uncorrelated' if best_model_info['residuals_ok'] else '⚠️ Autocorrelated'}")

# ============================================================
# 8. TRAINING SELECTED MODEL AND DIAGNOSTICS
# ============================================================

print("\n" + "="*60)
print("8. TRAINING SELECTED MODEL AND DIAGNOSTICS")
print("="*60)

print(f"\n🔧 Training SARIMA{best_order}×{best_seasonal_order}...")

sarima_model = SARIMAX(
    depos_train,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_results = sarima_model.fit(disp=False)

print("✅ Model trained")

# 8.1. Model summary
print("\n📊 Model summary:")
print(sarima_results.summary())

# 8.2. Residual diagnostics
print("\n🔍 SARIMA residual diagnostics:")

residuals_sarima = sarima_results.resid

# Ljung-Box test
lb_test = acorr_ljungbox(residuals_sarima, lags=[4, 8, 12], return_df=True)
print("\n📊 Ljung-Box test (autocorrelation):")
print(lb_test.round(4))

lb_min_p = lb_test['lb_pvalue'].min()
if lb_min_p > 0.05:
    print(f"   Conclusion: ✅ No autocorrelation (all p > 0.05)")
else:
    print(f"   Conclusion: ⚠️ Autocorrelation remains (minimum p = {lb_min_p:.4f})")
    print(f"   → Significant lags require additional AR/MA components")
    print(f"   → Iterative parameter addition recommended")

# Diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Diagnostics of SARIMA{best_order}×{best_seasonal_order}', fontsize=14)

axes[0, 0].plot(residuals_sarima.index, residuals_sarima, linewidth=1)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0, 0].set_title('Model residuals')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(residuals_sarima, bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=1)
axes[0, 1].set_title('Residual distribution')
axes[0, 1].set_xlabel('Residuals')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

plot_acf(residuals_sarima, lags=24, ax=axes[1, 0])
axes[1, 0].set_title('ACF of residuals')
axes[1, 0].set_xlabel('Lag')
axes[1, 0].set_ylabel('Autocorrelation')

from scipy import stats
stats.probplot(residuals_sarima, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q plot')

plt.tight_layout()
plt.savefig('05_sarima_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Diagnostics completed")

# ============================================================
# 9. FORECAST AND COMPARISON
# ============================================================

print("\n" + "="*60)
print("9. FORECAST AND COMPARISON")
print("="*60)

forecast_test = sarima_results.forecast(steps=len(depos_test))
fitted_values = sarima_results.fittedvalues

r2_train_sarima = r2_score(depos_train, fitted_values)
rmse_train_sarima = np.sqrt(mean_squared_error(depos_train, fitted_values))

r2_test_sarima = r2_score(depos_test, forecast_test)
rmse_test_sarima = np.sqrt(mean_squared_error(depos_test, forecast_test))
mae_test_sarima = mean_absolute_error(depos_test, forecast_test)

print(f"\n📊 SARIMA{best_order}×{best_seasonal_order}:")
print(f"   TRAINING sample (n={len(depos_train)}):")
print(f"     R²_train = {r2_train_sarima:.4f}")
print(f"     RMSE_train = {rmse_train_sarima:.2f} billion RUB")
print(f"   TEST sample (n={len(depos_test)}):")
print(f"     R²_test = {r2_test_sarima:.4f}")
print(f"     RMSE_test = {rmse_test_sarima:.2f} billion RUB")
print(f"     MAE_test = {mae_test_sarima:.2f} billion RUB")

print(f"\n📊 Comparison with full Ridge model (38 features, block 4):")
print(f"   {'Model':<35} {'R²_test':<10} {'RMSE_test':<12} {'MAE_test':<12}")
print(f"   {'-'*70}")
print(f"   {'Full Ridge (38 features)':<35} {'0.9422':<10} {'566.09':<12} {'489.89':<12}")
print(f"   {f'SARIMA{best_order}×{best_seasonal_order}':<35} {r2_test_sarima:<10.4f} {rmse_test_sarima:<12.2f} {mae_test_sarima:<12.2f}")

# Forecast visualization
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['DEPOS'], label='Actual data', color='#1f77b4', linewidth=2.5)
plt.plot(depos_test.index, forecast_test, label=f'Forecast SARIMA{best_order}×{best_seasonal_order}',
         color='#ff7f0e', linestyle='--', linewidth=2.5)

forecast_result = sarima_results.get_forecast(steps=len(depos_test))
confidence_intervals = forecast_result.conf_int(alpha=0.05)

plt.fill_between(
    depos_test.index,
    confidence_intervals.iloc[:, 0],
    confidence_intervals.iloc[:, 1],
    alpha=0.25, color='#ff7f0e', label='95% confidence interval'
)

plt.title(f'Deposit Volume Forecast — SARIMA{best_order}×{best_seasonal_order}\n' +
          f'R²_test = {r2_test_sarima:.4f}, RMSE = {rmse_test_sarima:.2f} billion RUB',
          fontsize=14)
plt.xlabel('Date')
plt.ylabel('Deposit volume, billion RUB')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_sarima_forecast.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 10. SARIMAX WITH EXOGENOUS VARIABLES
# ============================================================

print("\n" + "="*60)
print("10. SARIMAX WITH EXOGENOUS VARIABLES")
print("="*60)

print("""
📌 IDEA: Add macroeconomic indicators as exogenous variables.
This will allow the model to account for external factors in forecasting.

Advantages:
- Accounts for influence of WAGE, CPI, UNEM and other factors
- May better adapt to structural changes
- Combines strengths of SARIMA and Ridge
""")

# Prepare exogenous variables
# Use the same features as in Ridge model (block 4)
exog_features = ['WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP']

# Create lags for exogenous variables
for col in exog_features:
    for lag in [1, 3, 6]:
        df[f'{col}_lag_{lag}'] = df[col].shift(lag)

# Add seasonal dummy variables
df['Month'] = df.index.month
for month in range(2, 13):
    df[f'month_{month}'] = (df['Month'] == month).astype(int)

# Add structural break post_2022
df['post_2022'] = (df.index >= '2023-01-01').astype(int)

# Form complete set of exogenous variables
exog_columns = [col for col in df.columns if col not in [
    'DEPOS', 'DEPOS_log', 'DEPOS_diff1', 'DEPOS_diff2', 'Month'
]]

print(f"\n📊 Total exogenous variables: {len(exog_columns)}")
print(f"   Base: {len(exog_features)}")
print(f"   Lags: {len(exog_features) * 3}")
print(f"   Seasonal: 11")
print(f"   Structural: 1 (post_2022)")

# Prepare train/test for exogenous variables
exog_full = df[exog_columns].dropna()
common_index = depos_train.index.intersection(exog_full.index)

exog_train = exog_full.loc[common_index]
depos_train_aligned = depos_train.loc[common_index]

# For test period take last 12 records
exog_test = exog_full.iloc[-12:]

print(f"\n📊 Training sample (with exogenous): {len(depos_train_aligned)} records")
print(f"📊 Test sample (with exogenous): {len(exog_test)} records")
print(f"📊 Training period: {depos_train_aligned.index[0]} — {depos_train_aligned.index[-1]}")
print(f"📊 Test period: {exog_test.index[0]} — {exog_test.index[-1]}")

# Train SARIMAX with exogenous variables
print(f"\n🔧 Training SARIMAX{best_order}×{best_seasonal_order} with exogenous...")
print(f"   This may take about a minute...")

try:
    sarimax_model = SARIMAX(
        depos_train_aligned,
        exog=exog_train,
        order=best_order,
        seasonal_order=best_seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    sarimax_results = sarimax_model.fit(disp=False, maxiter=200)

    print("✅ SARIMAX model trained")

    # Forecast for test period
    forecast_sarimax = sarimax_results.forecast(steps=len(depos_test), exog=exog_test)

    # Test metrics
    r2_test_sarimax = r2_score(depos_test, forecast_sarimax)
    rmse_test_sarimax = np.sqrt(mean_squared_error(depos_test, forecast_sarimax))
    mae_test_sarimax = mean_absolute_error(depos_test, forecast_sarimax)

    # Training metrics
    fitted_sarimax = sarimax_results.fittedvalues
    r2_train_sarimax = r2_score(depos_train_aligned, fitted_sarimax)
    rmse_train_sarimax = np.sqrt(mean_squared_error(depos_train_aligned, fitted_sarimax))

    print(f"\n📊 SARIMAX with exogenous variables:")
    print(f"   TRAINING sample (n={len(depos_train_aligned)}):")
    print(f"     R²_train = {r2_train_sarimax:.4f}")
    print(f"     RMSE_train = {rmse_train_sarimax:.2f} billion RUB")
    print(f"   TEST sample (n={len(depos_test)}):")
    print(f"     R²_test = {r2_test_sarimax:.4f}")
    print(f"     RMSE_test = {rmse_test_sarimax:.2f} billion RUB")
    print(f"     MAE_test = {mae_test_sarimax:.2f} billion RUB")

    # SARIMAX residual diagnostics
    residuals_sarimax = sarimax_results.resid
    lb_test_sarimax = acorr_ljungbox(residuals_sarimax, lags=[4, 8, 12], return_df=True)
    lb_min_p_sarimax = lb_test_sarimax['lb_pvalue'].min()

    print(f"\n   RESIDUAL DIAGNOSTICS:")
    print(f"   Ljung-Box min p = {lb_min_p_sarimax:.4f}")
    print(f"   Residuals: {'✅ Uncorrelated' if lb_min_p_sarimax > 0.05 else '⚠️ Autocorrelated'}")

    # Comparison of all models
    print(f"\n📊 FINAL MODEL COMPARISON:")
    print(f"   {'Model':<35} {'R²_test':<10} {'RMSE':<12} {'MAE':<12} {'LB p':<10}")
    print(f"   {'-'*80}")
    print(f"   {'SARIMA (without exogenous)':<35} {r2_test_sarima:<10.4f} {rmse_test_sarima:<12.2f} {mae_test_sarima:<12.2f} {lb_min_p:<10.4f}")
    print(f"   {'SARIMAX (with exogenous)':<35} {r2_test_sarimax:<10.4f} {rmse_test_sarimax:<12.2f} {mae_test_sarimax:<12.2f} {lb_min_p_sarimax:<10.4f}")
    print(f"   {'Full Ridge (38 features, block 4)':<35} {'0.9422':<10} {'566.09':<12} {'489.89':<12} {'—':<10}")

    # Forecast visualization
    plt.figure(figsize=(14, 7))

    plt.plot(df.index, df['DEPOS'], label='Actual data', color='#1f77b4', linewidth=2.5)
    plt.plot(depos_test.index, forecast_test, label=f'SARIMA{best_order}×{best_seasonal_order}',
             color='#ff7f0e', linestyle='--', linewidth=2)
    plt.plot(depos_test.index, forecast_sarimax, label='SARIMAX (with exogenous)',
             color='#2ca02c', linestyle='--', linewidth=2)

    plt.title('Forecast Comparison: SARIMA vs SARIMAX', fontsize=14)
    plt.xlabel('Date')
    plt.ylabel('Deposit volume, billion RUB')
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('05_sarimax_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Visual analysis of forecasts
    print("\n📊 VISUAL ANALYSIS OF FORECASTS:")
    print(f"   Actual values (test): {depos_test.min():.0f} — {depos_test.max():.0f} billion RUB")
    print(f"   SARIMA forecast: {forecast_test.min():.0f} — {forecast_test.max():.0f} billion RUB")
    print(f"   SARIMAX forecast: {forecast_sarimax.min():.0f} — {forecast_sarimax.max():.0f} billion RUB")

    # Check overestimation/underestimation
    mean_actual = depos_test.mean()
    mean_sarima = forecast_test.mean()
    mean_sarimax = forecast_sarimax.mean()

    print(f"\n   Mean values:")
    print(f"   Actual: {mean_actual:.0f} billion RUB")
    print(f"   SARIMA: {mean_sarima:.0f} billion RUB ({(mean_sarima - mean_actual) / mean_actual * 100:+.1f}%)")
    print(f"   SARIMAX: {mean_sarimax:.0f} billion RUB ({(mean_sarimax - mean_actual) / mean_actual * 100:+.1f}%)")

    if abs(mean_sarimax - mean_actual) > abs(mean_sarima - mean_actual):
        print(f"\n   ⚠️ SARIMAX gives {'OVERESTIMATED' if mean_sarimax > mean_actual else 'UNDERESTIMATED'} forecasts")
        print(f"   → Exogenous variables distort forecast due to overfitting")
        print(f"   → 48 exogenous variables are too many for n=128")
    else:
        print(f"\n   ✅ SARIMAX forecasts are closer to actual than SARIMA")

    # Save SARIMAX results
    sarimax_metrics = {
        'r2_train': r2_train_sarimax,
        'rmse_train': rmse_train_sarimax,
        'r2_test': r2_test_sarimax,
        'rmse_test': rmse_test_sarimax,
        'mae_test': mae_test_sarimax,
        'lb_min_p': lb_min_p_sarimax,
        'forecast': forecast_sarimax
    }

    print(f"\n✅ SARIMAX analysis completed")
    print(f"   → SARIMAX does not improve forecast due to overfitting on 48 exogenous variables")
    print(f"   → Ridge (38 features) remains the best model")

except Exception as e:
    print(f"⚠️ SARIMAX error: {e}")
    print("   Possible causes:")
    print("   - Multicollinearity of exogenous variables")
    print("   - Insufficient degrees of freedom (too many parameters)")
    print("   - Convergence problems")
    print("   → Recommended to reduce number of exogenous variables")
    print("   → Or use only key ones: WAGE, CPI, UNEM, DEP1")
    sarimax_metrics = None

# ============================================================
# 11. COMBINED MODEL: Full Ridge + SARIMA
# ============================================================

print("\n" + "="*60)
print("11. COMBINED MODEL: Full Ridge + SARIMA")
print("="*60)

print("""
📌 IDEA: Combine strengths of both models.

Full Ridge (38 features, block 4) forecasts the trend, SARIMA corrects residuals
  Step 1: Restore full Ridge (38 features from block 4)
  Step 2: Calculate residuals: e = DEPOS_actual - DEPOS_ridge
  Step 3: SARIMA models the residuals
  Step 4: Final forecast = Ridge + SARIMA(residuals)
""")

print("\n" + "-"*60)
print("Step 1: Restoring full Ridge (38 features)")
print("-"*60)

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler

# Restore features from block 4
df_ridge = df.copy()

# DEPOS lags
df_ridge['DEPOS_log'] = np.log(df_ridge['DEPOS'])
for lag in [1, 3, 6, 12]:
    df_ridge[f'DEPOS_lag_{lag}'] = df_ridge['DEPOS'].shift(lag)

# Seasonal dummy variables
df_ridge['Month'] = df_ridge.index.month
for month in range(2, 13):
    df_ridge[f'month_{month}'] = (df_ridge['Month'] == month).astype(int)

# Structural variables
df_ridge['post_2022'] = (df_ridge.index >= '2023-01-01').astype(int)
df_ridge['covid'] = ((df_ridge.index >= '2020-03-01') & (df_ridge.index <= '2022-01-01')).astype(int)

# CRED1 regime
df_ridge['regime_cred1'] = (df_ridge['DEPOS'] > 32000).astype(int)

# WAGE anomalies
model_wage = LinearRegression()
model_wage.fit(df_ridge[['WAGE']].values, df_ridge['DEPOS'].values)
df_ridge['residual_wage'] = df_ridge['DEPOS'] - model_wage.predict(df_ridge[['WAGE']].values)
threshold_anomaly = -2.0 * df_ridge['residual_wage'].std()
df_ridge['anomaly_wage'] = (df_ridge['residual_wage'] < threshold_anomaly).astype(int)

# Macro factor lags
for col in ['WAGE', 'CPI', 'USDind']:
    for lag in [1, 3, 6]:
        df_ridge[f'{col}_lag_{lag}'] = df_ridge[col].shift(lag)

# Interaction UNEM × DEP1
df_ridge['UNEM_DEP1'] = df_ridge['UNEM'] * df_ridge['DEP1']

# Final feature set (38, as in block 4)
feature_columns_38 = [
    'WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP',
    'DEPOS_lag_1', 'DEPOS_lag_3', 'DEPOS_lag_6', 'DEPOS_lag_12',
    'month_2', 'month_3', 'month_4', 'month_5', 'month_6',
    'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12',
    'post_2022', 'covid', 'regime_cred1', 'anomaly_wage',
    'WAGE_lag_1', 'WAGE_lag_3', 'WAGE_lag_6',
    'CPI_lag_1', 'CPI_lag_3', 'CPI_lag_6',
    'USDind_lag_1', 'USDind_lag_3', 'USDind_lag_6',
    'UNEM_DEP1'
]

# Data preparation
X_ridge = df_ridge[feature_columns_38].dropna()
y_ridge = df_ridge.loc[X_ridge.index, 'DEPOS']

# Train/test split (122/12 as in block 4)
train_size_ridge = 122
X_train_ridge = X_ridge.iloc[:train_size_ridge]
X_test_ridge = X_ridge.iloc[train_size_ridge:]
y_train_ridge = y_ridge.iloc[:train_size_ridge]
y_test_ridge = y_ridge.iloc[train_size_ridge:]

# Scaling
scaler_ridge = StandardScaler()
X_train_scaled_ridge = scaler_ridge.fit_transform(X_train_ridge)
X_test_scaled_ridge = scaler_ridge.transform(X_test_ridge)

# Train Ridge
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled_ridge, y_train_ridge)

# Predictions
ridge_train_pred = ridge_model.predict(X_train_scaled_ridge)
ridge_test_pred = ridge_model.predict(X_test_scaled_ridge)

print(f"   ✅ Full Ridge trained: {X_ridge.shape[1]} features")
print(f"   R²_train = {r2_score(y_train_ridge, ridge_train_pred):.4f}")
print(f"   R²_test = {r2_score(y_test_ridge, ridge_test_pred):.4f}")
print(f"   RMSE_test = {np.sqrt(mean_squared_error(y_test_ridge, ridge_test_pred)):.2f}")
print(f"   MAE_test = {mean_absolute_error(y_test_ridge, ridge_test_pred):.2f}")

print("\n" + "-"*60)
print("Step 2: Calculating Ridge residuals")
print("-"*60)

residuals_ridge_train = y_train_ridge.values - ridge_train_pred
residuals_ridge_test = y_test_ridge.values - ridge_test_pred

print(f"   Residuals on training sample: {len(residuals_ridge_train)}")
print(f"   Residuals on test sample: {len(residuals_ridge_test)}")

print("\n" + "-"*60)
print("Step 3: SARIMA models Ridge residuals")
print("-"*60)
print("   Fitting optimal SARIMA for residuals...")
print("   This may take about a minute...")

# Convert residuals to Series with dates
residuals_ridge_train_series = pd.Series(
    residuals_ridge_train,
    index=y_train_ridge.index
)

# Fit SARIMA for residuals
best_residual_model = None
best_residual_aic = np.inf

for p in range(0, 3):
    for q in range(0, 3):
        for d_resid in range(0, 2):
            try:
                model_resid = SARIMAX(
                    residuals_ridge_train_series,
                    order=(p, d_resid, q),
                    seasonal_order=(0, 0, 0, 0),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                results_resid = model_resid.fit(disp=False)

                if results_resid.aic < best_residual_aic:
                    best_residual_aic = results_resid.aic
                    best_residual_model = results_resid
                    best_residual_order = (p, d_resid, q)
            except:
                continue

if best_residual_model is not None:
    print(f"   ✅ Best SARIMA for residuals: {best_residual_order}")
    print(f"   AIC = {best_residual_aic:.2f}")

    # Residual forecast
    residual_forecast = best_residual_model.forecast(steps=len(y_test_ridge))

    print("\n" + "-"*60)
    print("Step 4: Combined forecast")
    print("-"*60)

    combined_forecast = ridge_test_pred + residual_forecast.values

    # Combined model metrics
    r2_combined = r2_score(y_test_ridge, combined_forecast)
    rmse_combined = np.sqrt(mean_squared_error(y_test_ridge, combined_forecast))
    mae_combined = mean_absolute_error(y_test_ridge, combined_forecast)

    print(f"\n📊 Combined model (Full Ridge + SARIMA):")
    print(f"   TRAINING sample:")
    print(f"     R²_train = {r2_score(y_train_ridge, ridge_train_pred + best_residual_model.fittedvalues):.4f}")
    print(f"   TEST sample:")
    print(f"     R²_test = {r2_combined:.4f}")
    print(f"     RMSE_test = {rmse_combined:.2f} billion RUB")
    print(f"     MAE_test = {mae_combined:.2f} billion RUB")

    # Residual diagnostics
    combined_residuals = y_test_ridge.values - combined_forecast

    print(f"\n   RESIDUAL DIAGNOSTICS (test sample, n={len(combined_residuals)}):")

    try:
        lb_combined = acorr_ljungbox(combined_residuals, lags=[1, 2, 3, 4], return_df=True)
        lb_combined_min_p = lb_combined['lb_pvalue'].min()

        print(f"   Ljung-Box min p = {lb_combined_min_p:.4f}")
        print(f"   Residuals: {'✅ Uncorrelated' if lb_combined_min_p > 0.05 else '⚠️ Autocorrelated'}")
    except Exception as e:
        print(f"   ⚠️ Ljung-Box test not applicable: {e}")
        lb_combined_min_p = None

    # Final comparison
    print(f"\n📊 FINAL COMPARISON OF ALL MODELS:")
    print(f"   {'Model':<40} {'R²_test':<10} {'RMSE':<12} {'MAE':<12}")
    print(f"   {'-'*75}")
    print(f"   {'SARIMA (without exogenous)':<40} {r2_test_sarima:<10.4f} {rmse_test_sarima:<12.2f} {mae_test_sarima:<12.2f}")

    if sarimax_metrics is not None:
        print(f"   {'SARIMAX (48 exogenous)':<40} {sarimax_metrics['r2_test']:<10.4f} {sarimax_metrics['rmse_test']:<12.2f} {sarimax_metrics['mae_test']:<12.2f}")

    print(f"   {'Full Ridge (38 features)':<40} {r2_score(y_test_ridge, ridge_test_pred):<10.4f} {np.sqrt(mean_squared_error(y_test_ridge, ridge_test_pred)):<12.2f} {mean_absolute_error(y_test_ridge, ridge_test_pred):<12.2f}")
    print(f"   {'Combined (Ridge + SARIMA)':<40} {r2_combined:<10.4f} {rmse_combined:<12.2f} {mae_combined:<12.2f}")

    # Visualization
    plt.figure(figsize=(14, 7))

    plt.plot(df.index, df['DEPOS'], label='Actual data', color='#1f77b4', linewidth=2.5)
    plt.plot(y_test_ridge.index, ridge_test_pred, label='Full Ridge (38 features)',
             color='#ff7f0e', linestyle='--', linewidth=2)
    plt.plot(y_test_ridge.index, combined_forecast, label='Combined (Ridge + SARIMA)',
             color='#2ca02c', linestyle='--', linewidth=2)

    plt.title('Forecast Comparison: Full Ridge vs Combined Model', fontsize=14)
    plt.xlabel('Date')
    plt.ylabel('Deposit volume, billion RUB')
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('05_combined_model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Visual analysis
    print("\n📊 VISUAL ANALYSIS OF COMBINED MODEL:")

    mean_actual = y_test_ridge.mean()
    mean_ridge = ridge_test_pred.mean()
    mean_combined = combined_forecast.mean()

    print(f"   Mean values (test):")
    print(f"   Actual: {mean_actual:.0f} billion RUB")
    print(f"   Ridge: {mean_ridge:.0f} billion RUB ({(mean_ridge - mean_actual) / mean_actual * 100:+.1f}%)")
    print(f"   Combined: {mean_combined:.0f} billion RUB ({(mean_combined - mean_actual) / mean_actual * 100:+.1f}%)")

    ridge_errors = np.abs(y_test_ridge.values - ridge_test_pred)
    combined_errors = np.abs(y_test_ridge.values - combined_forecast)

    print(f"\n   Mean absolute error:")
    print(f"   Ridge: {ridge_errors.mean():.0f} billion RUB")
    print(f"   Combined: {combined_errors.mean():.0f} billion RUB")

    if combined_errors.mean() < ridge_errors.mean():
        print(f"\n   ✅ Combined model IMPROVES Ridge forecast")
        print(f"   Improvement: {(ridge_errors.mean() - combined_errors.mean()) / ridge_errors.mean() * 100:.1f}%")
    else:
        print(f"\n   ⚠️ Combined model DOES NOT improve Ridge forecast")
        print(f"   Deterioration: {(combined_errors.mean() - ridge_errors.mean()) / ridge_errors.mean() * 100:.1f}%")

else:
    print("⚠️ Failed to fit SARIMA for Ridge residuals")


# ============================================================
# 12. SARIMAX WITH KEY EXOGENOUS VARIABLES
# ============================================================

print("\n" + "="*60)
print("12. SARIMAX WITH KEY EXOGENOUS VARIABLES")
print("="*60)

print("""
📌 IDEA: Use only key macroeconomic variables
to avoid overfitting (problem of section 10 with 48 variables).

Selection based on SHAP analysis (block 3) and Ridge coefficients (block 4).
Key variables:
- WAGE (salary) — strongest macroeconomic driver
- CPI (inflation) — negative impact on deposits
- UNEM (unemployment) — confirmed nonlinearity
- DEP1 (rates) — affects deposit attractiveness
- post_2022 (structural shift) — significant trend change
""")

# Form reduced set of exogenous variables
key_exog_features = ['WAGE', 'CPI', 'UNEM', 'DEP1']

# Add lags only for key variables (lag 1)
for col in key_exog_features:
    df[f'{col}_lag_1_key'] = df[col].shift(1)

# Final set: base + lag 1 + post_2022
key_exog_columns = [
    'WAGE', 'CPI', 'UNEM', 'DEP1',
    'WAGE_lag_1_key', 'CPI_lag_1_key', 'UNEM_lag_1_key', 'DEP1_lag_1_key',
    'post_2022'
]

print(f"\n📊 Reduced set of exogenous variables: {len(key_exog_columns)}")
print(f"   Base: 4 (WAGE, CPI, UNEM, DEP1)")
print(f"   Lags (1 month): 4")
print(f"   Structural: 1 (post_2022)")

# Prepare train/test
exog_key_full = df[key_exog_columns].dropna()
common_index_key = depos_train.index.intersection(exog_key_full.index)

exog_key_train = exog_key_full.loc[common_index_key]
depos_key_train = depos_train.loc[common_index_key]
exog_key_test = exog_key_full.iloc[-12:]

print(f"\n📊 Training sample: {len(depos_key_train)} records")
print(f"📊 Test sample: {len(exog_key_test)} records")

# Train SARIMAX with reduced set
print(f"\n🔧 Training SARIMAX{best_order}×{best_seasonal_order} with {len(key_exog_columns)} exogenous...")

try:
    sarimax_key_model = SARIMAX(
        depos_key_train,
        exog=exog_key_train,
        order=best_order,
        seasonal_order=best_seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    sarimax_key_results = sarimax_key_model.fit(disp=False, maxiter=200)

    print("✅ Model trained")

    # Forecast
    forecast_sarimax_key = sarimax_key_results.forecast(steps=len(depos_test), exog=exog_key_test)

    # Metrics
    r2_train_key = r2_score(depos_key_train, sarimax_key_results.fittedvalues)
    rmse_train_key = np.sqrt(mean_squared_error(depos_key_train, sarimax_key_results.fittedvalues))
    r2_test_key = r2_score(depos_test, forecast_sarimax_key)
    rmse_test_key = np.sqrt(mean_squared_error(depos_test, forecast_sarimax_key))
    mae_test_key = mean_absolute_error(depos_test, forecast_sarimax_key)

    # Residual diagnostics
    residuals_key = sarimax_key_results.resid
    lb_key = acorr_ljungbox(residuals_key, lags=[4, 8, 12], return_df=True)
    lb_key_min_p = lb_key['lb_pvalue'].min()

    print(f"\n📊 SARIMAX with {len(key_exog_columns)} key variables:")
    print(f"   TRAINING sample (n={len(depos_key_train)}):")
    print(f"     R²_train = {r2_train_key:.4f}")
    print(f"     RMSE_train = {rmse_train_key:.2f} billion RUB")
    print(f"   TEST sample:")
    print(f"     R²_test = {r2_test_key:.4f}")
    print(f"     RMSE = {rmse_test_key:.2f} billion RUB")
    print(f"     MAE = {mae_test_key:.2f} billion RUB")
    print(f"   DIAGNOSTICS:")
    print(f"     Ljung-Box min p = {lb_key_min_p:.4f}")
    print(f"     Residuals: {'✅ Uncorrelated' if lb_key_min_p > 0.05 else '⚠️ Autocorrelated'}")

    # Visualization
    plt.figure(figsize=(14, 7))
    plt.plot(df.index, df['DEPOS'], label='Actual data', color='#1f77b4', linewidth=2.5)
    plt.plot(depos_test.index, forecast_sarimax_key,
             label=f'SARIMAX ({len(key_exog_columns)} exogenous)',
             color='#d62728', linestyle='--', linewidth=2)
    plt.plot(depos_test.index, forecast_test,
             label=f'SARIMA{best_order}×{best_seasonal_order}',
             color='#ff7f0e', linestyle='--', linewidth=1.5, alpha=0.7)

    plt.title(f'SARIMAX Forecast with Key Variables\nR²_test = {r2_test_key:.4f}, RMSE = {rmse_test_key:.2f}',
              fontsize=14)
    plt.xlabel('Date')
    plt.ylabel('Deposit volume, billion RUB')
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('05_sarimax_key_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Update final comparison
    print(f"\n📊 UPDATED FINAL COMPARISON OF ALL MODELS:")
    print(f"   {'Model':<40} {'R²_test':<10} {'RMSE':<12} {'LB p':<10}")
    print(f"   {'-'*75}")
    print(f"   {'SARIMA (without exogenous)':<40} {r2_test_sarima:<10.4f} {rmse_test_sarima:<12.2f} {lb_min_p:<10.4f}")
    if sarimax_metrics is not None:
        print(f"   {'SARIMAX (48 exogenous)':<40} {sarimax_metrics['r2_test']:<10.4f} {sarimax_metrics['rmse_test']:<12.2f} {sarimax_metrics['lb_min_p']:<10.4f}")
    print(f"   {'SARIMAX (9 key)':<40} {r2_test_key:<10.4f} {rmse_test_key:<12.2f} {lb_key_min_p:<10.4f}")

    # Save results
    sarimax_key_metrics = {
        'r2_train': r2_train_key,
        'rmse_train': rmse_train_key,
        'r2_test': r2_test_key,
        'rmse_test': rmse_test_key,
        'mae_test': mae_test_key,
        'lb_min_p': lb_key_min_p,
        'forecast': forecast_sarimax_key
    }

    print(f"\n✅ SARIMAX with key variables completed")

except Exception as e:
    print(f"⚠️ Error: {e}")
    sarimax_key_metrics = None


# ============================================================
# 13. FINAL SUMMARY FOR BLOCK 5
# ============================================================

print("\n" + "="*60)
print("13. FINAL SUMMARY FOR BLOCK 5")
print("="*60)

print(f"""
📌 KEY RESULTS OF BLOCK 5 (FULL OVERVIEW):

1. STATIONARITY:
   - Original DEPOS series: non-stationary (d=2)
   - Annual seasonality (s=12) confirmed by ACF/PACF

2. MODELS:
   ├── SARIMA(3, 2, 3)×(0, 0, 1, 12):
   │   R²_test = {r2_test_sarima:.4f}, RMSE = {rmse_test_sarima:.2f}
   │   Residuals: ✅ Uncorrelated (LB p = {lb_min_p:.4f})
   │
   ├── SARIMAX (48 exogenous):
   │   R²_test = {sarimax_metrics['r2_test']:.4f}, RMSE = {sarimax_metrics['rmse_test']:.2f}
   │   Residuals: ⚠️ Autocorrelated (LB p = {sarimax_metrics['lb_min_p']:.4f})
   │   Problem: CRITICAL overfitting
   │
   ├── SARIMAX (9 key):
   │   R²_test = {sarimax_key_metrics['r2_test']:.4f}, RMSE = {sarimax_key_metrics['rmse_test']:.2f}
   │   Residuals: ⚠️ Autocorrelated (LB p = {sarimax_key_metrics['lb_min_p']:.4f})
   │   Problem: reducing variables did NOT solve overfitting
   │
   ├── Full Ridge (38 features, block 4):
   │   R²_test = 0.9422, RMSE = 566.09
   │   ✅ Proven model
   │
   └── Combined (Full Ridge + SARIMA):
       R²_test = {r2_combined:.4f}, RMSE = {rmse_combined:.2f}

3. MAIN CONCLUSIONS:
   - SARIMA solves the autocorrelation problem ✅
   - SARIMAX (48) critically overfits ❌
   - SARIMAX (9) still overfitted, forecasts overestimated ❌
   - Full Ridge (38 features) — best model for forecasting ✅
   - Combined model does not improve Ridge ❌

4. WHY SARIMAX DOES NOT WORK:
   - Even 9 key variables cause overfitting with n=133
   - Exogenous variables in SARIMAX require large data volumes
   - Structural shift 2022-2023 is not captured by time series models
   - Ridge better handles multicollinearity through regularization

5. RECOMMENDATIONS:
   - For forecasting: Full Ridge (38 features, block 4) — R²_test = 0.9422
   - For understanding time structure: SARIMA (residuals uncorrelated)
   - SARIMAX with exogenous NOT recommended for this dataset
   - Combined approach does not provide improvement

6. FINAL DECISION:
   - BLOCK 4 (Full Ridge) — final model for forecasting
   - BLOCK 5 (SARIMA) — diagnostic tool
   - SARIMAX rejected as unsuitable for small samples

7. VISUAL OBSERVATIONS:
   - SARIMAX (9 key) also gives overestimated forecasts
   - Reducing variables did not solve overfitting problem
   - Full Ridge and Combined model practically coincide with actual data
""")

print("✅ BLOCK 5 FULLY COMPLETED")